# Advanced Agent Concepts — Lab Setup

In this module we explore more advanced agent patterns: connecting agents to external tools via a standard protocol, delegating work between agents, structuring multi-agent workflows as graphs, and giving agents live access to the web.

## Module 4 Roadmap

The labs in this module build on each other in a deliberate progression:

| Lab | Pattern | What you build |
|-----|---------|---------------|
| **1 — MCP** | Standard tool protocol | An MCP server that exposes tools over a well-known interface, and an MCP client that connects to it |
| **2 — Multi-agent delegation** | Agents as tools | A supervisor agent that delegates sub-tasks to specialist agents, treating each agent as a callable tool |
| **3 — Multi-agent graph** | Structured orchestration | A stateful graph (using LangGraph-style routing) that coordinates multiple agents with explicit edges and checkpoints |
| **4 — Browser tool** | Real-world web interaction | An agent equipped with a browser tool that navigates live web pages to complete research tasks |

By the end of the module you will have built a **deep research agent** that combines all four patterns.

---

## Why Tavily?

The research agents in labs 1–3 need to retrieve live information from the internet. We use **Tavily** as the search backbone.

Tavily provides a search API that returns structured snippets and source URLs — not raw HTML. This means an LLM can reason directly over the results without needing to parse web pages. Each response contains a `results` list of ranked snippets, a `query` echo-back, and optionally a synthesized `answer` field. This gives agents **real-world grounding via live internet access** with minimal token overhead.

Tavily offers a generous free tier that is more than sufficient for all labs in this module.

### Setup steps

1. Go to [https://tavily.com/](https://tavily.com/) and create a free account.
2. Create a `.env` file in this `notebooks/` directory.
3. Add your Tavily API key to the file:

```bash
TAVILY_KEY=<YOUR_KEY_HERE>
```

---

## About `labs_common`

All notebooks in this module import from **`labs_common`**, a shared package that lives in this repository and is installed as a local dependency.

It provides two things:

- **Centralized model IDs** — `HAIKU_STRANDS_ID` and `SONNET_STRANDS_ID` are string constants pointing to the correct Amazon Bedrock model identifiers. Using these constants means that if a model ID changes, every notebook is updated in one place.
- **`LabsBasePrompt`** — a Pydantic model with four fields: `system_prompt` (str), `user_prompt` (str), `model_id` (str), and `hyperparams` (dict). Passing a `LabsBasePrompt` to an agent invocation gives a consistent, validated interface across every lab.

These abstractions let the notebooks focus on the **patterns being demonstrated** rather than boilerplate configuration.

In [ ]:
import os 
from dotenv import load_dotenv
# Check if you created the .env file before running this notebook.
print('Does the .env file exist?', os.path.exists('.env'))
# from dotenv import load_dotenv
load_dotenv('.env')

## Test the Tavily connection

The cell below fires a test search query. A **successful response** is a Python dict with the following shape:

```python
{
    "query": "What are the main differences...",   # echoes back your query
    "answer": "...",                               # optional: Tavily's synthesized answer
    "results": [                                   # list of ranked web snippets
        {
            "title": "...",
            "url": "https://...",
            "content": "...",                      # the snippet an LLM can reason over
            "score": 0.95,
        },
        ...
    ]
}
```

If you see a `results` list with at least one entry, your API key is working correctly and you are ready to proceed.

In [ ]:
# Test it out
from typing import Dict

from tavily import TavilyClient

client = TavilyClient(os.getenv("TAVILY_KEY"))

response: Dict = client.search(query="What are the main differences between how the US and EU handle milk pasteurization?")

print(response)

# Setup complete

The Tavily API key is working and you'll see it used throughout notebooks 1–3 as the external search capability powering our agents.

Next, continue to **notebook 1 — MCP**, where you'll build an MCP server that exposes Tavily as a tool and connect an MCP client to call it.